# Exploratory Data Analysis: Urban Flood Nowcasting

This notebook verifies the generated synthetic dataset and ensures the mathematical distributions of our mock elevations and drainage scores behave as expected before we integrate them into the backend risk model.

**Author:** [Placeholder Member B]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for plots
sns.set_theme(style="whitegrid")

# Load synthetic data generated by data/generate_dataset.py
df = pd.read_csv('../data/processed/zones.csv')
df.head()

## 1. Topographical Variance
Let's check the elevation distribution. We expect a mean around 560m.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['elevation_m'], bins=5, kde=True, color="skyblue")
plt.title('Distribution of Zone Elevations in Pune Ward')
plt.xlabel('Elevation (meters above sea level)')
plt.ylabel('Frequency')
plt.show()

print(f"Min Elevation: {df['elevation_m'].min()}m")
print(f"Max Elevation: {df['elevation_m'].max()}m")

## 2. Infrastructure Assessment
Visualizing the synthetic drainage capacity scores. Score 1 is poor, 5 is excellent.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='drainage_capacity_score', palette='viridis')
plt.title('Drainage Capacity Scores Across Zones')
plt.xlabel('Drainage Score (1-5)')
plt.ylabel('Number of Zones')
plt.show()

## 3. Initial Baseline Simulation
We will now import the backend `risk_model.py` directly into this notebook to ensure the risk bands make sense with our current baseline (low) rainfall.

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.abspath(''), '..', 'backend'))

from risk_model import calculate_risk

# Apply the model logic to our dataframe
def apply_model(row):
    res = calculate_risk(row['elevation_m'], row['drainage_capacity_score'], row['current_rainfall_mm'])
    return pd.Series([res['score'], res['band']])

df[['risk_score', 'risk_band']] = df.apply(apply_model, axis=1)
df[['name', 'elevation_m', 'drainage_capacity_score', 'current_rainfall_mm', 'risk_score', 'risk_band']]

**Conclusion:** The data distributions are sound. The baseline risk scores fall appropriately into the Low/Medium bands, setting up a stable environment for the live `/simulate` endpoint to inject severe rainfall.